# Notebook 01 — Multi-Agent Orchestration with LangGraph

## Objectives
- Build a multi-agent LangGraph pipeline from scratch
- Understand shared state (`TypedDict`) flowing between nodes
- Add conditional routing: if retrieval is empty, fall back to an API agent
- Learn the supervisor pattern: run N agents in parallel, pick the best result
- Use `day5.multi_agent_orchestrator` for a production-ready implementation

## 1. Simple inline LangGraph pipeline

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Optional

class AgentState(TypedDict):
    query: str
    rewritten: Optional[str]
    docs: list[str]
    answer: Optional[str]

def rewrite(s): return {"rewritten": s["query"] + " detailed"}
def retrieve(s): return {"docs": [f"Doc about: {s['rewritten'][:30]}"]}
def synthesize(s): return {"answer": f"Based on {len(s['docs'])} docs: {s['docs'][0][:50]}"}

g = StateGraph(AgentState)
g.add_node("rewrite",    rewrite)
g.add_node("retrieve",   retrieve)
g.add_node("synthesize", synthesize)
g.add_edge(START, "rewrite")
g.add_edge("rewrite",    "retrieve")
g.add_edge("retrieve",   "synthesize")
g.add_edge("synthesize", END)
pipeline = g.compile()

result = pipeline.invoke({"query": "What is hybrid search?", "rewritten": None, "docs": [], "answer": None})
print(result["answer"])

## What just happened?

We built a 3-node graph:
1. **rewrite** — appends `" detailed"` to the query
2. **retrieve** — returns a mock document based on the rewritten query
3. **synthesize** — combines docs into an answer

Each node receives the full `AgentState` dict and returns only the keys it changes.
LangGraph merges updates automatically — this is the reducer pattern.

## 2. Conditional routing: API fallback when retrieval is empty

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Optional

class RouterState(TypedDict):
    query: str
    docs: list[str]
    api_result: Optional[str]
    answer: Optional[str]

def retrieve_node(s):
    # Simulate empty retrieval for unknown queries
    if "unknown" in s["query"]:
        return {"docs": []}
    return {"docs": [f"Found doc about: {s['query']}"]}

def api_node(s):
    return {"api_result": f"API data for: {s['query']}"}

def synth_node(s):
    src = s.get("api_result") or (s["docs"][0] if s["docs"] else "nothing")
    return {"answer": f"Answer from: {src[:60]}"}

def route(s):
    return "api" if not s["docs"] else "synth"

g = StateGraph(RouterState)
g.add_node("retrieve", retrieve_node)
g.add_node("api",      api_node)
g.add_node("synth",    synth_node)
g.add_edge(START, "retrieve")
g.add_conditional_edges("retrieve", route, {"api": "api", "synth": "synth"})
g.add_edge("api",  "synth")
g.add_edge("synth", END)
g2 = g.compile()

r1 = g2.invoke({"query": "hybrid search",  "docs": [], "api_result": None, "answer": None})
r2 = g2.invoke({"query": "unknown topic",  "docs": [], "api_result": None, "answer": None})
print("Known query  :", r1["answer"])
print("Unknown query:", r2["answer"])

## 3. Supervisor vs Parallel agent patterns

Two common multi-agent patterns:

| Pattern | Description | When to use |
|---------|-------------|-------------|
| **Sequential** | Nodes pass state one-by-one | Dependent steps (rewrite → retrieve → synthesize) |
| **Supervisor** | Multiple agents run, supervisor picks best | Ensemble answers, confidence voting |
| **Parallel (map-reduce)** | Fan out → fan in | Independent sub-tasks (research multiple topics) |

LangGraph supports all three via `add_edge`, `add_conditional_edges`, and `Send`.

## 4. Parallel agents — supervisor picks best result

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class SuperState(TypedDict):
    task: str
    results: list[str]
    best: str

# Two "expert" agents
def agent_keyword(state):
    return {"results": [f"[keyword]: Found keyword match for '{state['task']}'" ]}

def agent_semantic(state):
    return {"results": [f"[semantic]: Found semantic match for '{state['task']}' — this is a more detailed semantic answer"]}

def supervisor(state):
    best = max(state["results"], key=len)  # pick most detailed
    return {"best": best}

# Note: in real LangGraph parallel execution you'd use Send() or separate branches
# For simplicity here we call both agents sequentially and let supervisor choose
def run_both(state):
    kw  = f"[keyword]: Found keyword match for '{state['task']}'"
    sem = f"[semantic]: Found semantic match for '{state['task']}' — more detailed semantic answer"
    best = max([kw, sem], key=len)
    return {"results": [kw, sem], "best": best}

g = StateGraph(SuperState)
g.add_node("run_agents", run_both)
g.add_edge(START, "run_agents")
g.add_edge("run_agents", END)
supervisor_graph = g.compile()

result = supervisor_graph.invoke({"task": "What is BM25?", "results": [], "best": ""})
print("All results:")
for r in result["results"]:
    print(f"  {r}")
print("\nBest:", result["best"])

## 5. Setup sys.path to use day5 modules

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

## 6. Using day5.multi_agent_orchestrator

In [ ]:
from day5.multi_agent_orchestrator import build_orchestrator, run_orchestrator

# Sample knowledge base
docs = [
    "LangGraph enables stateful multi-agent workflows with conditional routing",
    "BM25 is a keyword-based ranking algorithm used in information retrieval",
    "Hybrid search combines sparse BM25 and dense semantic embeddings",
    "RRF (Reciprocal Rank Fusion) merges multiple ranked lists into one",
]

def my_retrieve(query):
    return [d for d in docs if any(w in d.lower() for w in query.lower().split())][:2]

graph = build_orchestrator(retrieve_fn=my_retrieve)
result = run_orchestrator(graph, "How does hybrid search work?")

print("Query         :", result["query"])
print("Rewritten     :", result["rewritten_query"])
print("Docs retrieved:", len(result["retrieved_docs"]))
print("Synthesis     :", result["synthesis"])
print("Cost tokens   :", result["cost_tokens"])

## 7. Using the supervisor orchestrator

In [ ]:
from day5.multi_agent_orchestrator import build_supervisor_orchestrator

# Define two specialist agents
agents = [
    ("retrieval_agent", lambda task: f"Retrieved 3 docs about: {task}"),
    ("web_agent",       lambda task: f"Web search found: multiple articles discussing {task} with additional context and depth"),
]

sup_graph = build_supervisor_orchestrator(agents)
result = sup_graph.invoke({"task": "LangGraph multi-agent", "agent_results": [], "final_answer": ""})

print("Agent results:")
for r in result["agent_results"]:
    print(f"  {r}")
print("\nFinal answer:", result["final_answer"])